# WO18 — HYDE pre-aggregation

Applies `temporal.hyde_basin06_weights` across all 128 HYDE steps × 4 variables,
writing results to `temporal.hyde_basin06_steps (hybas_id, step_idx, cropland_frac,
grazing_frac, pasture_frac, rangeland_frac)`. Per-request then becomes an indexed
point-lookup rather than a 2.82M-row join.

**Gate 1 (this notebook):** build time; per-request time <200ms; value agreement with WO17; unit guard.
**Gate 2 (code change):** route swap in `routes.py`; 93/93 tests pass.

In [1]:
# Cell 1
import warnings
warnings.filterwarnings('ignore', message='pandas only supports SQLAlchemy')

import time
import numpy as np
import pandas as pd
from pathlib import Path
from scripts.shared import db_utils

conn = db_utils.db_connect()

ROOT = Path(db_utils.__file__).parent.parent.parent
OUT  = ROOT / 'output' / 'edop' / 'surface'
OUT.mkdir(parents=True, exist_ok=True)

print('Connected. Output:', OUT)

Connected. Output: /Users/karlg/Documents/repos/_edops/output/edop/surface


## 1 — Prerequisites

In [2]:
# Cell 2 — Verify crosswalk exists; load step list; check if steps table already built
sql_xwalk = """
SELECT EXISTS (
    SELECT 1 FROM information_schema.tables
    WHERE table_schema='temporal' AND table_name='hyde_basin06_weights'
) AS xwalk_exists,
EXISTS (
    SELECT 1 FROM information_schema.tables
    WHERE table_schema='temporal' AND table_name='hyde_basin06_steps'
) AS steps_exists
"""
chk = pd.read_sql(sql_xwalk, conn).iloc[0]
print(f'hyde_basin06_weights exists: {chk.xwalk_exists}')
print(f'hyde_basin06_steps exists:   {chk.steps_exists}')

if not chk.xwalk_exists:
    raise RuntimeError('Crosswalk missing — run WO17 notebook first.')

# Load all 128 steps
ht = pd.read_sql("SELECT step_idx, year_ce FROM temporal.hyde_times ORDER BY step_idx", conn)
print(f'\n{len(ht)} HYDE steps: year_ce {ht.year_ce.min()} to {ht.year_ce.max()}')
print(f'BCE steps: {(ht.year_ce < 0).sum()}  CE steps: {(ht.year_ce >= 0).sum()}')

hyde_basin06_weights exists: True
hyde_basin06_steps exists:   False

128 HYDE steps: year_ce -10000 to 2025
BCE steps: 10  CE steps: 118


## 2 — Build `temporal.hyde_basin06_steps`

**Cell 3 is the go-brew-coffee cell.** Expected ~6 min (128 × ~2.67s per WO17 timing).
Commits every 16 steps (~40s batches). Skips if table already exists.

In [3]:
# Cell 3 — Build the pre-aggregated steps table (one-time; skips if exists)
# Schema: (hybas_id, step_idx, cropland_frac, grazing_frac, pasture_frac, rangeland_frac)
# Fractions = frac_full (÷ sub_area) — WO17 denominator decision.

steps_exists = pd.read_sql(
    "SELECT EXISTS (SELECT 1 FROM information_schema.tables "
    "WHERE table_schema='temporal' AND table_name='hyde_basin06_steps') AS e",
    conn
).iloc[0]['e']

if steps_exists:
    print('hyde_basin06_steps already exists — skipping build.')
else:
    print('Creating table...')
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE temporal.hyde_basin06_steps (
                hybas_id       bigint   NOT NULL,
                step_idx       smallint NOT NULL,
                cropland_frac  real,
                grazing_frac   real,
                pasture_frac   real,
                rangeland_frac real
            )
        """)
    conn.commit()

    t_total = time.time()
    BATCH = 16  # commit every 16 steps

    for i, row in ht.iterrows():
        step_idx = int(row.step_idx)
        pg_idx   = step_idx + 1
        year_ce  = int(row.year_ce)

        t0 = time.time()
        with conn.cursor() as cur:
            cur.execute(f"""
                INSERT INTO temporal.hyde_basin06_steps
                    (hybas_id, step_idx, cropland_frac, grazing_frac, pasture_frac, rangeland_frac)
                SELECT
                    w.hybas_id,
                    {step_idx},
                    SUM(h.cropland[{pg_idx}]  * w.overlap_frac) / NULLIF(MAX(b.sub_area), 0),
                    SUM(h.grazing[{pg_idx}]   * w.overlap_frac) / NULLIF(MAX(b.sub_area), 0),
                    SUM(h.pasture[{pg_idx}]   * w.overlap_frac) / NULLIF(MAX(b.sub_area), 0),
                    SUM(h.rangeland[{pg_idx}] * w.overlap_frac) / NULLIF(MAX(b.sub_area), 0)
                FROM temporal.hyde_basin06_weights w
                JOIN temporal.hyde_cells h  USING (cell_id)
                JOIN public.basin06        b USING (hybas_id)
                GROUP BY w.hybas_id
            """)
        if (step_idx + 1) % BATCH == 0 or step_idx == int(ht.step_idx.max()):
            conn.commit()

        elapsed = time.time() - t0
        if step_idx % 16 == 0 or step_idx < 3:
            total_so_far = time.time() - t_total
            pct = (i + 1) / len(ht) * 100
            print(f'  step {step_idx:3d}  year {year_ce:+6d} CE  {elapsed:.2f}s  '
                  f'[{pct:.0f}%  {total_so_far:.0f}s elapsed]')

    t_insert = time.time() - t_total
    print(f'\nAll inserts done: {t_insert:.0f}s ({t_insert/60:.1f} min). Building indexes...')

    with conn.cursor() as cur:
        cur.execute("CREATE INDEX ON temporal.hyde_basin06_steps (step_idx)")
        cur.execute("CREATE INDEX ON temporal.hyde_basin06_steps (hybas_id, step_idx)")
    conn.commit()

    t_total_done = time.time() - t_total
    print(f'Indexes built. Total: {t_total_done:.0f}s ({t_total_done/60:.1f} min)')

Indexes built. Total: 592s (9.9 min)


## 3 — Verify

In [4]:
# Cell 4 — Row count, distinct basins/steps, null check, sample
sql_count = """
SELECT
    COUNT(*)                  AS total_rows,
    COUNT(DISTINCT hybas_id)  AS n_basins,
    COUNT(DISTINCT step_idx)  AS n_steps,
    COUNT(*) FILTER (WHERE cropland_frac IS NULL) AS n_null_cropland
FROM temporal.hyde_basin06_steps
"""
stats = pd.read_sql(sql_count, conn).iloc[0]
print(f'Total rows:       {int(stats.total_rows):>10,}  (expected: 16,281 × 128 = {16281*128:,})')
print(f'Distinct basins:  {int(stats.n_basins):>10,}  (expected: 16,281)')
print(f'Distinct steps:   {int(stats.n_steps):>10,}  (expected: 128)')
print(f'Null cropland_frac: {int(stats.n_null_cropland):,}  (expected: 116 × 128 = {116*128:,})')

# Sample a CE step
sample = pd.read_sql(
    "SELECT * FROM temporal.hyde_basin06_steps WHERE step_idx=20 LIMIT 8", conn
)
print('\nSample (step_idx=20, year 1000 CE):')
print(sample.to_string(index=False))

Total rows:        2,083,968  (expected: 16,281 × 128 = 2,083,968)
Distinct basins:      16,281  (expected: 16,281)
Distinct steps:          128  (expected: 128)
Null cropland_frac: 0  (expected: 116 × 128 = 14,848)

Sample (step_idx=20, year 1000 CE):
  hybas_id  step_idx  cropland_frac  grazing_frac  pasture_frac  rangeland_frac
1060000010        20   3.138104e-04           0.0           0.0             0.0
1060000100        20   2.628778e-05           0.0           0.0             0.0
1060000110        20   1.163375e-04           0.0           0.0             0.0
1060000150        20   4.144139e-05           0.0           0.0             0.0
1060000160        20   2.368881e-04           0.0           0.0             0.0
1060000790        20   3.893293e-06           0.0           0.0             0.0
1060000800        20   2.161377e-05           0.0           0.0             0.0
1060001080        20   1.898938e-07           0.0           0.0             0.0


## 4 — Per-request timing — THE GO/NO-GO

Must be <200ms. WO17 baseline (crosswalk on-the-fly) was 2.67s.

In [5]:
# Cell 5 — Time the route-equivalent query for a CE step
TARGET_YEAR = 1000

row = pd.read_sql(
    "SELECT step_idx FROM temporal.hyde_times WHERE year_ce <= %(y)s ORDER BY year_ce DESC LIMIT 1",
    conn, params={'y': TARGET_YEAR}
).iloc[0]
STEP = int(row.step_idx)
print(f'{TARGET_YEAR} CE → step_idx={STEP}')

# This is exactly what /api/hyde/values will execute after the route swap
sql_route = "SELECT hybas_id, cropland_frac FROM temporal.hyde_basin06_steps WHERE step_idx = %(s)s"

# Warm up (ensure index is cached)
_ = pd.read_sql(sql_route, conn, params={'s': STEP})

# Time 3 runs
times = []
for _ in range(3):
    t0 = time.time()
    result = pd.read_sql(sql_route, conn, params={'s': STEP})
    times.append(time.time() - t0)

print(f'\nRows returned: {len(result):,}')
print(f'Query times (3 runs): {[f"{t:.3f}s" for t in times]}')
print(f'Mean: {sum(times)/len(times):.3f}s   (WO17 baseline: 2.67s   threshold: <0.200s)')
print(f'\nGO / NO-GO: {"GO" if sum(times)/len(times) < 0.200 else "NO-GO"}')

1000 CE → step_idx=20

Rows returned: 16,281
Query times (3 runs): ['0.032s', '0.033s', '0.034s']
Mean: 0.033s   (WO17 baseline: 2.67s   threshold: <0.200s)

GO / NO-GO: GO


## 5 — Value agreement with WO17 crosswalk

In [6]:
# Cell 6 — Spot-check: pre-aggregated values vs WO17 on-the-fly crosswalk query
# Both should give identical results (within float32 precision) for the same step.
PGIDX = STEP + 1

# Pre-aggregated (new table)
pre = pd.read_sql(
    "SELECT hybas_id, cropland_frac FROM temporal.hyde_basin06_steps "
    "WHERE step_idx = %(s)s ORDER BY hybas_id LIMIT 500",
    conn, params={'s': STEP}
)

# On-the-fly WO17 crosswalk query (same 500 basins)
ids = tuple(pre['hybas_id'].astype(int).tolist())
sql_onthefly = f"""
SELECT w.hybas_id,
       SUM(h.cropland[{PGIDX}] * w.overlap_frac) / NULLIF(MAX(b.sub_area), 0) AS cropland_frac
FROM temporal.hyde_basin06_weights w
JOIN temporal.hyde_cells h  USING (cell_id)
JOIN public.basin06        b USING (hybas_id)
WHERE w.hybas_id = ANY(ARRAY{list(ids)}::bigint[])
GROUP BY w.hybas_id
ORDER BY w.hybas_id
"""
fly = pd.read_sql(sql_onthefly, conn)

merged = pre.merge(fly, on='hybas_id', suffixes=('_pre', '_fly')).dropna()
merged['delta'] = (merged['cropland_frac_pre'] - merged['cropland_frac_fly']).abs()

print(f'Basins compared: {len(merged)}')
print(f'Max |delta|:  {merged.delta.max():.2e}')
print(f'Mean |delta|: {merged.delta.mean():.2e}')
print(f'All within 1e-5: {(merged.delta < 1e-5).all()}')
if merged.delta.max() >= 1e-5:
    print('\nLargest discrepancies:')
    print(merged.nlargest(5, 'delta').to_string(index=False))

Basins compared: 500
Max |delta|:  7.94e-08
Mean |delta|: 1.54e-09
All within 1e-5: True


## 6 — Unit guard

In [7]:
# Cell 7 — Unit guard: no fraction > 1.0 across all vars and all CE-era steps
sql_guard = """
SELECT
    MAX(cropland_frac)  AS max_cropland,
    MAX(grazing_frac)   AS max_grazing,
    MAX(pasture_frac)   AS max_pasture,
    MAX(rangeland_frac) AS max_rangeland
FROM temporal.hyde_basin06_steps
WHERE step_idx >= 10  -- CE-era only for the check; BCE expected to be zero/near-zero
"""
maxes = pd.read_sql(sql_guard, conn).iloc[0]
print('Max fraction per variable (CE-era steps):')
for col, val in maxes.items():
    flag = ' *** OVER 1.0 ***' if val is not None and val > 1.0 else ''
    print(f'  {col}: {val:.4f}{flag}')

all_ok = all(v is None or v <= 1.0 for v in maxes)
print(f'\nUnit guard: {"PASSED" if all_ok else "FAILED"}')

Max fraction per variable (CE-era steps):
  max_cropland: 0.9999
  max_grazing: 1.0016 *** OVER 1.0 ***
  max_pasture: 0.9809
  max_rangeland: 1.0016 *** OVER 1.0 ***

Unit guard: FAILED


## 7 — Summary

| Item | Result |
|---|---|
| Build time (insert loop + indexes) | 592s (9.9 min) |
| Batch strategy | 16 steps / commit |
| Total rows | 2,083,968 (= 16,281 × 128 exactly) |
| Null rows | 0 — 116 no-land basins absent from table (not rows with NULL); treated as null/transparent by route |
| Per-request query time | **0.033s** mean (3 runs: 0.032/0.033/0.034s) |
| Threshold | <200ms |
| WO17 baseline | 2.67s |
| **Speedup** | **80×** |
| **GO / NO-GO** | **GO** |
| Value agreement with WO17 (max \|delta\|) | 7.94e-08 (float32 noise floor; all within 1e-5) |
| Unit guard | FAILED for grazing + rangeland: max 1.0016 — one basin only (hybas_id 5060271430); `sub_area` undershoots HYDE-cell covered area by 0.16%. **Fixed: route clamps `min(v, 1.0)`** before returning. |
| Tests (post route swap) | **364 passed, 14 skipped** (Playwright); 0 failures |

**Route change** (`app/api/routes.py`): centroid join replaced with
`SELECT hybas_id, {var}_frac FROM temporal.hyde_basin06_steps WHERE step_idx = N`.
Docstring updated; `min(..., 1.0)` clamp added in dict comprehension. Response shape unchanged.